In [1]:
import httpx
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
import time
import json
import psycopg
import os
import sys
import numpy as np
from dotenv import load_dotenv

sys.path.append(os.path.abspath('./src'))
import db_functions as dbf
load_dotenv()
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST", "localhost")
port = os.getenv("DB_PORT", "5432")
dbname = os.getenv("DB_NAME")
conn_str = f"postgresql://{user}:{password}@{host}:{port}/{dbname}"
api_key = os.getenv("API_KEY")
api_url = 'https://api.stratz.com/graphql'
headers = {
    'User-Agent': 'STRATZ_API',
    "Authorization": f"Bearer {api_key}"
}

In [ ]:
# with psycopg.connect(conn_str) as conn:
#     with conn.cursor() as cur:
#         result = cur.execute("SELECT table_name FROM information_schema.tables").fetchall()
#         for table_name in result:
#             if table_name[0].startswith('match_'):
#                 cur.execute(f'DROP TABLE {table_name[0]}')

In [15]:
## Parsing matches to dataframes and to sql tables
with psycopg.connect(conn_str) as conn:
    with conn.cursor() as cur:
        match_id_tups = cur.execute("SELECT match_id FROM matches WHERE start_date >= '2025-02-21 00:00:00' ORDER BY start_date ASC;").fetchall()
        match_ids = [tup[0] for tup in match_id_tups]
        parsed_id_tups = cur.execute("SELECT id FROM match_details").fetchall()
        parsed_ids = [tup[0] for tup in parsed_id_tups]
        unfinished_match_ids = []
        for match_id in match_ids:
            if match_id not in parsed_ids:
                unfinished_match_ids.append(match_id)


In [ ]:
nested_vars = []
for key, value in result_json.items():
    if type(value) in [dict, list]:
        print(key, type(value))
        nested_vars.append(key)

pickBans <class 'list'>
chatEvents <class 'list'>
predictedWinRates <class 'list'>
winRates <class 'list'>
radiantNetworthLeads <class 'list'>
radiantExperienceLeads <class 'list'>
radiantKills <class 'list'>
direKills <class 'list'>
towerDeaths <class 'list'>
towerStatus <class 'list'>
players <class 'list'>


In [ ]:
for key, value in result_json['players'][0].items():
    if type(value) in [dict, list]:
        print(key, type(value))
        nested_vars.append(key)

stats <class 'dict'>


In [25]:
for key, value in result_json['players'][0]['stats'].items():
    if type(value) in [dict, list]:
        print(key, type(value))
        nested_vars.append(key)

impPerMinute <class 'list'>
goldPerMinute <class 'list'>
networthPerMinute <class 'list'>
experiencePerMinute <class 'list'>
towerDamagePerMinute <class 'list'>
campStack <class 'list'>
deathEvents <class 'list'>
farmDistributionReport <class 'dict'>
matchPlayerBuffEvent <class 'list'>
inventoryReport <class 'list'>
itemPurchases <class 'list'>
courierKills <class 'list'>
runes <class 'list'>
wards <class 'list'>
wardDestruction <class 'list'>


In [4]:
mid_query = 'SELECT DISTINCT match_id FROM match_players WHERE match_players."steamAccountId" IS NULL'
matches = dbf.query_select_to_df(conn_str, mid_query, 'match_players', columns=['match_id'])
query = '''
    query($id: Long!) {
        match(id: $id) {
            players {
                heroId
                steamAccountId
                partyId
                steamAccount {
                    name
                    realName
                    profileUri
                    timeCreated
                    isAnonymous
                    proSteamAccount {
                        teamId
                        name
                    }
                }
            }
        }
    }
'''
with psycopg.connect(conn_str) as conn:
    with conn.cursor() as cur:
        for match_id in matches['match_id']:
            result = dbf.query_stratz(query, headers, api_url, variables={'id': match_id})
            res = result['data']['match']['players']
            df_players_addition = pd.DataFrame(res)
            df_steam_account = pd.json_normalize(df_players_addition['steamAccount'])
            og_cols = df_steam_account.columns
            new_cols = [str.replace(colname, '.', '_') for colname in df_steam_account.columns]
            col_mapper = {og_col: new_col for og_col, new_col in zip(og_cols, new_cols)}
            df_steam_account = df_steam_account.rename(col_mapper, axis=1)
            df_players_addition = df_players_addition.drop('steamAccount', axis=1)
            df_players_final = pd.concat([df_players_addition, df_steam_account], axis=1)
            try:
                df_players_final = df_players_final.drop('proSteamAccount', axis=1) #Sometimes this column gets left there empty
            except:
                pass
            for idx, row in df_players_final.iterrows():
                for col in df_players_final.columns:
                    if pd.isna(row[col]):
                        cur.execute(f'UPDATE match_players SET "{col}" = NULL WHERE match_players.match_id = %s AND match_players."heroId" = %s;', (match_id, row['heroId']))
                    else:
                        cur.execute(f'UPDATE match_players SET "{col}" = %s WHERE match_players.match_id = %s AND match_players."heroId" = %s;', (row[col], match_id, row['heroId']))
            conn.commit()
            

In [34]:
with psycopg.connect(conn_str) as conn:
    cols = []
    for col_name, dtype in zip(df_players_final.columns, df_players_final.dtypes):
        if col_name != 'heroId':
            pg_type = dbf.get_pg_type(dtype)
            # Wrap column names in quotes to handle spaces or reserved words
            cols.append(f'ADD COLUMN"{col_name}" {pg_type}')
    
    schema = ", ".join(cols)
    create_table_query = f'ALTER TABLE "match_players" {schema};'
    with conn.cursor() as cur:
        cur.execute(create_table_query)

In [8]:
queries = [
    'SELECT id FROM league_details ld WHERE ld."displayName" LIKE \'ESL%\' AND ld."prizePool" <> 0;',
    'SELECT id FROM league_details ld WHERE ld."displayName" LIKE \'%DreamLeague%\' AND ld."prizePool" <> 0;',
    'SELECT id FROM league_details ld WHERE ld."displayName" LIKE \'%International%\' AND ld."prizePool" <> 0;',
    'SELECT id FROM league_details ld WHERE ld."displayName" LIKE \'FISSURE%\' AND ld."displayName" NOT LIKE \'%Special\' AND ld."prizePool" <> 0;',
    'SELECT id FROM league_details ld WHERE ld."displayName" LIKE \'%Clavision%\' AND ld."prizePool" <> 0;'
]

In [19]:
league_ids = pd.DataFrame()
for query in queries:
    batch_ids = dbf.query_select_to_df(conn_str, query, 'league_details', columns=['id'])['id'] 
    league_ids = pd.concat([league_ids, batch_ids])
query = '''
    query($id: Int!, $request: LeagueMatchesRequestType!) {
        league(id: $id) {
            matches(request: $request) {
                id
            }
        }
    }
'''
league_ids = league_ids['id']
match_ids = pd.DataFrame(columns=['id'])
for league_id in league_ids:
    skip_counter = 0
    while True:
        results = dbf.query_stratz(
            query, 
            headers, 
            api_url, 
            variables={
                'id': int(league_id), 
                'request': {'isParsed': True, 'take':100, 'skip': skip_counter}})
        results_matches = results['data']['league']['matches']
        skip_counter += 100
        if len(results_matches) == 0:
            break
        match_ids = pd.concat([match_ids, pd.DataFrame(results_matches)], axis=0)
match_ids = match_ids.reset_index().drop(['index'], axis=1)
query = 'SELECT id FROM match_details;'
db_match_ids = dbf.query_select_to_df(conn_str, query, 'match_details', columns=['id'])
for idx, row in match_ids.copy().iterrows():
    if row['id'] in list(db_match_ids['id']):
        match_ids = match_ids.drop(idx, axis=0)

In [2]:
query = 'SELECT id FROM league_details ld WHERE ld."prizePool" > 500000;'
league_ids = dbf.query_select_to_df(conn_str, query, 'league_details', columns=['id'])['id']
query = '''
    query($id: Int!, $request: LeagueMatchesRequestType!) {
        league(id: $id) {
            matches(request: $request) {
                id
            }
        }
    }
'''
match_ids = pd.DataFrame(columns=['id'])
for league_id in league_ids:
    skip_counter = 0
    while True:
        results = dbf.query_stratz(
            query, 
            headers, 
            api_url, 
            variables={
                'id': league_id, 
                'request': {'isParsed': True, 'take':100, 'skip': skip_counter}})
        results_matches = results['data']['league']['matches']
        skip_counter += 100
        if len(results_matches) == 0:
            break
        match_ids = pd.concat([match_ids, pd.DataFrame(results_matches)], axis=0)
match_ids = match_ids.reset_index().drop(['index'], axis=1)
query = 'SELECT id FROM match_details;'
db_match_ids = dbf.query_select_to_df(conn_str, query, 'match_details', columns=['id'])
for idx, row in match_ids.copy().iterrows():
    if row['id'] in list(db_match_ids['id']):
        match_ids = match_ids.drop(idx, axis=0)

In [21]:
match_ids.to_csv('backup.csv')

In [2]:
match_ids = pd.read_csv('backup.csv')
query = 'SELECT id FROM match_details;'
db_match_ids = dbf.query_select_to_df(conn_str, query, 'match_details', columns=['id'])
for idx, row in match_ids.copy().iterrows():
    if row['id'] in list(db_match_ids['id']):
        match_ids = match_ids.drop(idx, axis=0)
match_ids = match_ids.reset_index()
match_ids

,index,Unnamed: 0,id
0,173,444,6158170178
1,174,445,6158081326
2,175,446,6158010330
3,176,447,6157905423
4,177,448,6157810302
...,...,...,...
4043,4216,5804,7868858167
4044,4217,5805,7868771308
4045,4218,5806,7868764939
4046,4219,5807,7868706982


In [4]:
match_ids = match_ids.drop(range(0, 499), axis=0)
match_ids

,index,Unnamed: 0,id
499,672,943,5645454664
500,673,944,5644626111
501,674,945,5644524719
502,675,946,5644420232
503,676,947,5644313409
...,...,...,...
4043,4216,5804,7868858167
4044,4217,5805,7868771308
4045,4218,5806,7868764939
4046,4219,5807,7868706982


In [3]:
dbf.query_match(conn_str, headers, api_url, list(match_ids['id']))

Currently processing iteration 0
Currently processing iteration 1
Currently processing iteration 2
Currently processing iteration 3
Currently processing iteration 4
Currently processing iteration 5
Currently processing iteration 6
Currently processing iteration 7
Currently processing iteration 8
Currently processing iteration 9
Currently processing iteration 10
Currently processing iteration 11
Currently processing iteration 12
Currently processing iteration 13
Currently processing iteration 14
Currently processing iteration 15
Currently processing iteration 16
Currently processing iteration 17
Currently processing iteration 18
Currently processing iteration 19
Currently processing iteration 20
Currently processing iteration 21
Currently processing iteration 22
Currently processing iteration 23
Currently processing iteration 24
Currently processing iteration 25
Currently processing iteration 26
Currently processing iteration 27
Currently processing iteration 28
Currently processing ite

KeyError: 'data'